# F6ii — Reward magnitude: attractor geometry → learning dynamics

**Headline.** Reward magnitude changes how fast the 3-neuron PC network learns, not the level it
learns to (main Fig. 6e). Both halves come from the **realized input drive**
$\langle x_0\rangle = s \times P(\text{optimal})$, which is simultaneously

* the **slope of the line attractor** in the $(w_2-w_1,\,x_2-x_1)$ plane (Panel C3), and
* the **gate on plasticity**, since the PC update is $\Delta w = \beta\,x_0\,e$ (Panel C4a).

Feeding that one number into a one-dimensional drift–diffusion process along the attractor
reproduces the whole family of learning curves on held-out seeds (Panel C4c), including their common
asymptote. Learning stops once reward no longer depends on the activity noise, which happens at a fixed
*margin* $m=\text{slope}\cdot W \approx 0.02$ for every reward, so every reward ends at the same
performance (C4d).

Operating point $(\alpha,\beta,\eta) = (0.0848, 0.0164, 0.003)$; the asymptote is 0.983–0.994 at
every reward. the companion paper (Li & Erlich) uses the same simulations, seeds and estimators
(`companion_paper/`).

Main-figure panels written by this notebook: **6e** `svgs/f6ii_mag_learningcurves.svg`,
**6f** `svgs/f6ii_mag_crosssections.svg`, **6g** `svgs/f6ii_mag_drift_diffusion.svg`,
**6h** `svgs/f6ii_mag_fp_reconstruction.svg`.

In [ ]:
import os
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401  (registers '3d')

from ssb.envs import bandits
from ssb.envs.bandits import bandit_det
from ssb.models import ssb, rules
from ssb.scripts import run
from ssb.utils.batch_inits import make_default_hps, batchify_hps, init_params_batch

from matplotlib import font_manager as _fm
mpl.rcParams['font.family'] = ('Arial' if any(f.name == 'Arial' for f in _fm.fontManager.ttflist)
                               else 'Liberation Sans')   # Liberation Sans is metric-identical to Arial
mpl.rcParams['figure.dpi'] = 200   # crisper inline figures; saved SVGs are unaffected

# side legends sit outside the axes, which tight_layout warns about but handles correctly
# once bbox_inches='tight' is used at save time.
import warnings
warnings.filterwarnings('ignore', message='.*not compatible with tight_layout.*')

SVG_DIR   = 'svgs'
CACHE_DIR = 'cache'
os.makedirs(SVG_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ----- house style (matches F1/F2 notebooks) -----
def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    """Nature-style 2D axes: no top/right spine, few ticks, 7pt ticks / 9pt labels."""
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'):
        ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)


def side_legend(ax, title=None, **kw):
    ax.legend(fontsize=7, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0),
              title=title, title_fontsize=7, handlelength=1.2, **kw)


# ----- colour: reward magnitude is a MAGNITUDE variable, so a single-hue
# ----- light->dark sequential ramp, identical to F2_magrate.ipynb's reward_color.
REWARD_CMAP = plt.cm.Blues

def reward_color(j, n):
    """Sequential ramp position for reward level j of n (same as F2)."""
    return REWARD_CMAP(0.35 + 0.55 * j / max(n - 1, 1))


def reward_color_abs(r, lo=0.2, hi=1.0):
    """Same sequential ramp as `reward_color`, anchored to the absolute reward value rather than
    to an index, so a given reward level keeps its colour across every panel in this notebook --
    including Panel C, which shows a different (and unevenly spaced) subset of reward levels."""
    return REWARD_CMAP(0.35 + 0.55 * (float(r) - lo) / (hi - lo))


def rolling_mean(x, w):
    """Non-overlapping bin mean (same as F1/F2): length len(x)//w."""
    n = (len(x) // w) * w
    return x[:n].reshape(-1, w).mean(axis=1)

## CONFIG — operating point, and why

**$(\alpha,\beta)$.** $\alpha = 0.0848$, $\beta = 0.0164$ is a top-scoring cell of the `data/det`
sweep for PC/small at $\eta=10^{-2}$ (score 1.000; the score saturates, so several cells tie). It has the timescale separation
$\beta/\alpha = 0.19$ that the slow-manifold reduction of `f6i_phase.ipynb` needs.

**$\eta$.** The asymptote is reward-independent only if the activity noise accumulated off the
attractor, $\sigma_{\rm OU}=\sqrt2\,\eta/\sqrt{1-(1-2\alpha)^2}$, is small compared with the margin
$sW$ that a learned weight produces at the smallest reward. Here
$\sigma_{\rm OU}=0.0076$. $\eta=0.003$ is between the
sweep's grid levels ($10^{-3}$, $10^{-2}$); it is a deliberate choice, and the diagnostics cell
below records the alternatives.

**Horizon.** The slowest reward ($s=0.2$) reaches criterion at ~120k trials and its plateau by
~250k, so every run is 300k trials.

In [ ]:
from pathlib import Path
from ssb.analysis.best_params import DET_FOLDER, best_params
from ssb.analysis.cell_select import ALPHAS, BETAS, ETAS

# ---- CONFIG ----
RULE  = 'PC'          # the predictive-coding rule; the attractor story is specific to it
SIZE  = 'small'       # the 3-neuron reduction f6i_phase derives; Panel C's coordinates need it
ALPHA = float(ALPHAS[11])      # 0.08483
BETA  = float(BETAS[9])        # 0.01638
ETA_A = 0.003

TIMESTEPS = 300_000
N_SEEDS   = 80
N_SAMPLES = 10        # independent 80-seed batches, for error bars on Panel B
REWARDS   = np.array([0.2, 0.3, 0.5, 0.7, 1.0])
WINDOW    = 100       # bin width for P(optimal) curves (same as F1/F2)
CRIT_FRAC = 0.9       # main-text criterion: first bin reaching 0.9 x that level's own asymptote

print(f'{RULE}/{SIZE} used here: alpha={ALPHA:.4g}  beta={BETA:.4g}  eta={ETA_A:g}  '
      f'beta/alpha={BETA/ALPHA:.3f}  1/(2 alpha)={1/(2*ALPHA):.1f} trials')
SIGMA_OU = np.sqrt(2) * ETA_A / np.sqrt(1 - (1 - 2 * ALPHA) ** 2)
print(f'sigma_OU = {SIGMA_OU:.4f}')

try:   # det-sweep score of this cell at eta = 1e-2 (needs data/det)
    x = np.load(Path(DET_FOLDER) / f'det_{RULE}_{SIZE}.npz')['x']
    score = np.median(x.mean(axis=-3)[..., -20000:, :, :], axis=-3)
    ie = int(np.argmin(np.abs(np.log(ETAS) - np.log(1e-2))))
    ia, ib = 11, 9
    print(f'det-sweep score at eta=1e-2: this cell {score[ie, ia, ib]:.3f}; '
          f'max {score[ie].max():.3f}; cells at the max: {(score[ie] == score[ie].max()).sum()}')
    del x
except FileNotFoundError as e:
    print('data/det not found, sweep score not printed:', e)

### Simulator

One hand-rolled step loop serves every panel, so arbitrary internals can be logged and every network
("seed") has its own PRNG key from `init_params`. Same step order as `ssb.scripts.run.run_env` with
`settle_time=1` (env → stimulus → activity step → noise and rectification → PC weight step), and, like
`run.run_quick`, **no activity cap** (activities never exceed ~1.4 here). The loop runs in 50k-trial
chunks and either bins choices (`window`) or stores every `stride`-th trial, so 300k-trial runs fit in
memory.

In [ ]:
from jax import vmap


def make_rows(seeds, alpha=None, beta=None, eta=None):
    n = len(seeds)
    hps = make_default_hps(0)
    hps['tau'] = jnp.array([0.])
    hb = batchify_hps(hps, batch_size=n)
    col = lambda v: jnp.broadcast_to(jnp.asarray(v, dtype=jnp.float32), (n,)).reshape(n, 1)
    hb['alpha'] = col(ALPHA if alpha is None else alpha)
    hb['beta'] = col(BETA if beta is None else beta)
    hb['eta_a'] = col(ETA_A if eta is None else eta)
    hb['seed'] = jnp.asarray(seeds).reshape(n, 1)
    return hb


def simulate(seeds, reward, timesteps, alpha=None, beta=None, eta=None, window=None, stride=None,
             chunk=50_000):
    """window: per-seed P(optimal) in non-overlapping bins of `window` trials -> out['B'] (bins, seeds).
    stride: (W, x0, X, A) every `stride`-th trial -> out['W'] etc. (records, seeds); A = 1 if the
    paying arm was chosen. out['W0'] is the initial weight difference."""
    hb = make_rows(seeds, alpha, beta, eta)
    n = len(seeds)
    upd_w, upd_s = rules.rule2fns(RULE)
    ua, uw = vmap(ssb.update_acts), vmap(upd_w)
    an, us = vmap(ssb.act_noise), vmap(upd_s)
    envb, expb = vmap(bandit_det), vmap(ssb.expand_stimulus)
    env_par = jnp.stack([jnp.zeros(n), jnp.full((n,), float(reward), jnp.float32)], 1)

    def step(state, _):
        es, acts, st, w, tern, key = state
        rew, act, es, key = envb(es, acts[:, -2:], env_par, key)
        stim = expb(rew, acts)
        acts = ua(stim, acts, st, w, tern, hb)
        acts, key = an(acts, key, hb)
        w = uw(stim, acts, st, w, tern, hb)
        st = us(acts, st, hb)
        return (es, acts, st, w, tern, key), (act, acts[:, 0], acts[:, 2] - acts[:, 1],
                                              w[:, 0, 2] - w[:, 0, 1], acts.max(1))

    @jax.jit
    def run_chunk(state):
        state, (act, x0, X, W, amax) = jax.lax.scan(step, state, None, length=chunk)
        outs = {'amax': amax.max(0)}
        if window:
            outs['B'] = act.reshape(chunk // window, window, n).mean(1)
        if stride:
            sl = slice(stride - 1, None, stride)
            outs['W'], outs['x0'], outs['X'], outs['A'] = W[sl], x0[sl], X[sl], act[sl]
        return state, outs

    acts, st, w, tern, key = init_params_batch(hb)
    state = (jnp.zeros((n, 2)), acts, st, w, tern, key)
    W0 = np.asarray(w[:, 0, 2] - w[:, 0, 1])
    res = {k: [] for k in ('B', 'W', 'x0', 'X', 'A')}
    amax = 0.
    for _ in range(timesteps // chunk):
        state, o = run_chunk(state)
        amax = max(amax, float(np.asarray(o['amax']).max()))
        for k in res:
            if k in o:
                res[k].append(np.asarray(o[k]))
    out = {k: np.concatenate(v, 0).astype(np.float32) for k, v in res.items() if v}
    if 'A' in out:
        out['A'] = (out['A'] == 1).astype(np.float32)
    out['W0'], out['amax'] = W0.astype(np.float32), amax
    return out

### Diagnostics: why this point and not the alternatives

Long-run behaviour (300k trials, 48 seeds) at the smallest and largest reward, for this
$(\alpha,\beta)$ at the sweep's noise level $\eta=10^{-2}$ and at the $\eta$ used here. At
$\eta=10^{-2}$ the asymptote still depends on reward (Supplementary Methods S1). About a minute.

In [ ]:
DIAG = {'this (alpha, beta) at eta = 0.01': (ALPHA, BETA, 0.01),
        'used here (0.0848, 0.0164, 0.003)': (ALPHA, BETA, ETA_A)}
print(f"{'':36} {'s=0.2: t90':>11} {'end':>6}   {'s=1.0: t90':>11} {'end':>6}")
for name, (a, b, e) in DIAG.items():
    row = []
    for s in (0.2, 1.0):
        B = simulate(np.arange(2000, 2048), s, 300_000, alpha=a, beta=b, eta=e, window=500)['B'].mean(1)
        A = B[-60:].mean(); p0 = B[:2].mean()
        t90 = (np.argmax(np.convolve(B, np.ones(10) / 10, 'same') >= p0 + 0.9 * (A - p0)) + 1) * 500
        row += [t90, A]
    print(f'{name:36} {row[0]:11,d} {row[1]:6.3f}   {row[2]:11,d} {row[3]:6.3f}')

## Panels A & B — behaviour

Same runs feed both. Panel A (**main Fig. 6e**) shows the **mean**-across-seeds $P(\text{optimal})$
curve per reward magnitude; Panel B extracts a rate and an asymptote from those curves. The mean is
used because the median is exactly 1.000 at every reward once more than half the seeds converge, and
because the mean is the statistic C4c's Fokker–Planck integral predicts. Seeds are the same as before:
10 batches of 80, seed base $235 + 9173k$.

In [ ]:
def run_reward_sweep(rewards, timesteps=TIMESTEPS, n_seeds=N_SEEDS, n_samples=N_SAMPLES,
                     window=WINDOW, seed0=235, use_cache=True):
    """Binned per-seed P(optimal): curves (n_rewards, n_samples, n_bins, n_seeds)."""
    rewards = np.asarray(rewards, dtype=float)
    tag = f'{RULE}_{SIZE}_a{ALPHA:.4g}_b{BETA:.4g}_e{ETA_A:.4g}_T{timesteps}_S{n_seeds}x{n_samples}_w{window}'
    path = os.path.join(CACHE_DIR, f'f6ii_sweep_{tag}.npz')
    if use_cache and os.path.exists(path):
        d = np.load(path)
        if np.allclose(d['rewards'], rewards):
            print(f'loaded cached sweep ({path})')
            return d['rewards'], d['curves']
    seeds = np.concatenate([seed0 + 9173 * k + np.arange(n_seeds) for k in range(n_samples)])
    curves = []
    for r in rewards:
        o = simulate(seeds, r, timesteps, window=window)
        curves.append(o['B'].reshape(-1, n_samples, n_seeds).transpose(1, 0, 2))
        print(f'reward {r:.2f} done ({n_samples} x {n_seeds} seeds, {timesteps:,} steps; max activity {o["amax"]:.2f})')
    curves = np.stack(curves).astype(np.float32)
    np.savez_compressed(path, rewards=rewards, curves=curves)
    return rewards, curves


REWARDS, CURVES = run_reward_sweep(REWARDS)
T_AXIS = (np.arange(CURVES.shape[2]) + 1) * WINDOW
print('curves:', CURVES.shape, '(rewards, samples, bins, seeds)')

In [ ]:
# ---- Panel A (main Fig. 6e): P(optimal) vs trial, one line per reward magnitude ----
fig, ax = plt.subplots(figsize=(2.6, 1.35))
for i, r in enumerate(REWARDS):
    ax.plot(T_AXIS, CURVES[i].mean(axis=-1).mean(axis=0), color=reward_color_abs(r), lw=0.9)
ax.axhline(0.5, color='0.85', lw=0.4, zorder=0)
nature_ax(ax, 'Trial', 'P(optimal)')
ax.set_xlim(0, T_AXIS[-1]); ax.set_ylim(0.45, 1.02)
ax.xaxis.set_major_locator(mticker.MultipleLocator(100_000))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x else '0'))
side_legend(ax, title='Reward', handles=
    [plt.Line2D([], [], color=reward_color_abs(r), lw=1.0, label=f'{float(r):.1f}') for r in REWARDS])
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_learningcurves.svg', dpi=300, bbox_inches='tight')
plt.show()

## Panel B — rate and asymptote vs reward magnitude

The main-text definitions (Methods, "Rate and asymptote sensitivity"), applied to the mean curve of
each 80-seed batch: asymptote $A$ = mean over the final 10% of bins; time to criterion = first 100-trial
bin at which the curve reaches $0.9A$; rate = $10^4$ / time to criterion. Because every reward reaches
the same asymptote here, the relative criterion is well conditioned. Open circles: the 10
batches; line: mean ± s.e.m.

In [ ]:
def curve_metrics(curve, window=WINDOW, crit_frac=CRIT_FRAC):
    asym = float(curve[-len(curve) // 10:].mean())
    t_crit = (int(np.argmax(curve >= crit_frac * asym)) + 1) * window
    return 1e4 / t_crit, asym, t_crit


rates, asyms, tcrits = (np.zeros(CURVES.shape[:2]) for _ in range(3))
for i in range(len(REWARDS)):
    for k in range(CURVES.shape[1]):
        rates[i, k], asyms[i, k], tcrits[i, k] = curve_metrics(CURVES[i, k].mean(axis=-1))

fig, axes = plt.subplots(1, 2, figsize=(4.2, 1.35), gridspec_kw=dict(wspace=0.6))
for ax, vals, ylab in [(axes[0], rates, 'Rate\n' r'($10^4$ / trials to criterion)'),
                       (axes[1], asyms, 'Asymptote')]:
    for i, r in enumerate(REWARDS):
        jit = np.random.default_rng(i).normal(0, 0.012, size=vals.shape[1])
        ax.scatter(np.full(vals.shape[1], float(r)) + jit, vals[i], facecolors='none',
                   edgecolors=reward_color_abs(r), s=7, lw=0.4, alpha=0.8, zorder=2)
    m, sem = vals.mean(1), vals.std(1) / np.sqrt(vals.shape[1])
    ax.errorbar(REWARDS.astype(float), m, yerr=sem, fmt='-', color='0.15', lw=0.9,
                marker='o', markersize=2.6, capsize=1.6, elinewidth=0.7, zorder=3)
    nature_ax(ax, 'Reward size', ylab, nx=5, ny=4)
axes[0].set_ylim(0, None)
axes[1].set_ylim(0.5, 1.02); axes[1].axhline(1.0, color='0.85', lw=0.4, zorder=0)
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_rate_asymptote.svg', dpi=300, bbox_inches='tight')
plt.show()

print(f"{'s':>5} {'t_crit':>16} {'rate':>16} {'asymptote':>17}")
for i, r in enumerate(REWARDS):
    se = lambda v: v.std() / np.sqrt(len(v))
    print(f'{r:5.2f} {tcrits[i].mean():8,.0f} ± {se(tcrits[i]):5,.0f} {rates[i].mean():7.3f} ± {se(rates[i]):.3f}'
          f' {asyms[i].mean():8.4f} ± {se(asyms[i]):.4f}')
print(f'rate ~ s^{np.polyfit(np.log(REWARDS), np.log(rates.mean(1)), 1)[0]:.2f}; '
      f'asymptote range {asyms.mean(1).min():.4f}-{asyms.mean(1).max():.4f}')

## Panel C — attractor geometry, and the dynamics along it

Difference coordinates $(x_{\rm in},\; W=w_2-w_1,\; X=x_2-x_1)$, matching `f6i_phase.ipynb`'s
reduction. The line attractor is $X = x_0 W$; it is marginally stable, with a zero eigenvalue along
$W$. Behaviour is pinned to the attractor coordinate, and learning is transport along the marginal
direction.

**Attractor runs.** 200 seeds (0–199) × 300k trials per reward, every 10th trial stored. Seeds 0–119
estimate $v$, $D$ and $P(m)$; 120–199 are held out. The steady-state samples for C1–C3 are the last
50k trials of these runs, by which point every reward has converged.

In [ ]:
C4_REWARDS = [0.2, 0.3, 0.5, 0.7, 1.0]
C4_SEEDS, C4_NFIT = 200, 120
C4_STEPS, C4_STRIDE = 300_000, 10
LAG_STEPS = 400                       # >> activity correlation time 1/(2 alpha) ~ 6 trials
LAG = LAG_STEPS // C4_STRIDE
SS_RECORDS = 50_000 // C4_STRIDE      # steady-state window: last 50k trials


def cross_section_slope(xi, W, X):
    """Through-origin least-squares slope of the W>0 cross-section in the central-50% x_in band."""
    lo, hi = np.percentile(xi, [25, 75])
    m = (xi >= lo) & (xi <= hi) & (W > 0)
    return float(np.sum(W[m] * X[m]) / np.sum(W[m] * W[m])), m


def run_attractor(rewards, use_cache=True):
    tag = f'{RULE}_{SIZE}_a{ALPHA:.4g}_b{BETA:.4g}_e{ETA_A:.4g}_T{C4_STEPS}_S{C4_SEEDS}_st{C4_STRIDE}'
    path = os.path.join(CACHE_DIR, f'f6ii_attractor_{tag}.npz')
    if use_cache and os.path.exists(path):
        d = np.load(path)
        print(f'loaded cached attractor runs ({path})')
        return {float(r): {k: d[f'{k}{i}'] for k in ('W', 'x0', 'X', 'A')} for i, r in enumerate(rewards)}
    out, blobs = {}, {}
    for i, r in enumerate(rewards):
        o = simulate(np.arange(C4_SEEDS), r, C4_STEPS, stride=C4_STRIDE)
        out[float(r)] = {k: o[k] for k in ('W', 'x0', 'X', 'A')}
        blobs.update({f'{k}{i}': o[k] for k in ('W', 'x0', 'X', 'A')})
        print(f'reward {r:.2f}: final P = {o["A"][-5000:].mean():.3f}; max activity {o["amax"]:.2f}')
    np.savez_compressed(path, rewards=np.asarray(rewards, float), **blobs)
    return out


ATT = run_attractor(C4_REWARDS)
for r in C4_REWARDS:
    xi, W, X = (ATT[r][k][-SS_RECORDS:].ravel() for k in ('x0', 'W', 'X'))
    ATT[r]['slope'], msk = cross_section_slope(xi, W, X)
    ATT[r]['drive'] = float(xi[msk].mean())
    ATT[r]['wdrive'] = float(np.sum(W[msk] ** 2 * xi[msk]) / np.sum(W[msk] ** 2))

In [ ]:
# ---- C1/C2: steady-state clouds (last 50k trials, first 60 seeds) ----
CLOUD_REWARDS = [0.2, 0.5, 1.0]
clouds = {r: tuple(ATT[r][k][-SS_RECORDS:, :60].ravel() for k in ('x0', 'W', 'X')) for r in CLOUD_REWARDS}


def _style3d(ax):
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.fill = False; pane.set_edgecolor('#CCCCCC'); pane.set_linewidth(0.25)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_major_locator(mticker.MaxNLocator(3)); axis.line.set_linewidth(0.25)
    ax.grid(False)
    ax.tick_params(labelsize=7, pad=-2, colors='#555555', width=0.25)
    ax.set_xlabel('$x_{in}$', fontsize=9, labelpad=-5)
    ax.set_ylabel('$w_2-w_1$', fontsize=9, labelpad=-5)
    ax.set_zlabel('$x_2-x_1$', fontsize=9, labelpad=-3)


def draw_clouds(elev=16, azim=-60, planes=False, n_show=2000, save=None):
    rng = np.random.default_rng(0)
    fig = plt.figure(figsize=(3.3, 2.6)); ax = fig.add_subplot(111, projection='3d')
    for r in CLOUD_REWARDS:
        xi, W, X = clouds[r]
        idx = rng.choice(len(xi), min(n_show, len(xi)), replace=False)
        ax.scatter(xi[idx], W[idx], X[idx], s=3, c=[reward_color_abs(r)], alpha=0.5,
                   linewidths=0, rasterized=True, label=f'$s$={r}')
    if planes:
        Whi = np.percentile(np.concatenate([clouds[r][1] for r in CLOUD_REWARDS]), 99)
        Xhi = np.percentile(np.concatenate([clouds[r][2] for r in CLOUD_REWARDS]), 99)
        WW, XX = np.meshgrid([0.0, Whi], [0.0, Xhi])
        for r in CLOUD_REWARDS:
            c = float(np.median(clouds[r][0]))
            ax.plot_surface(np.full_like(WW, c), WW, XX, color=reward_color_abs(r), alpha=0.20,
                            linewidth=0, shade=False)
            ax.plot([c] * 5, [0, Whi, Whi, 0, 0], [0, 0, Xhi, Xhi, 0], color=reward_color_abs(r), lw=1.4)
    ax.legend(fontsize=7, frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0), markerscale=3)
    _style3d(ax); ax.view_init(elev=elev, azim=azim)
    fig.subplots_adjust(left=0.04, right=0.74, top=1.0, bottom=0.0)   # not bbox 'tight': it crops the z-label
    if save:
        plt.savefig(save, dpi=300)
    plt.show()


draw_clouds(planes=False, save=f'{SVG_DIR}/f6ii_mag_clouds.svg')          # C1
draw_clouds(planes=True,  save=f'{SVG_DIR}/f6ii_mag_clouds_planes.svg')   # C2

In [ ]:
# ---- C3 (main Fig. 6f): cross-sections overlaid, through-origin fit ----
fig, ax = plt.subplots(figsize=(2.6, 1.75))
for r in CLOUD_REWARDS:
    xi, W, X = (ATT[r][k][-SS_RECORDS:].ravel() for k in ('x0', 'W', 'X'))
    slope, m = cross_section_slope(xi, W, X)
    sub = np.random.default_rng(1).choice(np.flatnonzero(m), min(30_000, m.sum()), replace=False)
    ax.scatter(W[sub], X[sub], s=2, c=[reward_color_abs(r)], alpha=0.03, linewidths=0, rasterized=True)
    xs = np.linspace(0, np.percentile(W[m], 99), 20)
    ax.plot(xs, slope * xs, color=reward_color_abs(r), lw=1.5, label=f'$s$={r}:  {slope:.2f}')
ax.axhline(0, color='0.85', lw=0.5, zorder=0); ax.axvline(0, color='0.85', lw=0.5, zorder=0)
nature_ax(ax, '$w_2-w_1$', '$x_2-x_1$')
side_legend(ax, title='cross-section slope')
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_crosssections.svg', dpi=300, bbox_inches='tight')
plt.show()

print(f"{'s':>5} {'slope':>8} {'<x_0>':>8} {'W^2-wtd <x_0>':>14} {'slope/s':>8}")
for r in C4_REWARDS:
    a = ATT[r]
    print(f"{r:5.2f} {a['slope']:8.3f} {a['drive']:8.3f} {a['wdrive']:14.3f} {a['slope']/r:8.3f}")

# During learning the slope is the drive s * P(optimal): binned by m = s W, <x_0>/s equals P.
print('\nduring learning, <x_0>/s vs P(optimal) in bins of m = s*W (fit seeds, all times):')
_e = np.array([-0.2, -0.05, -0.02, -0.01, 0, 0.01, 0.02, 0.05])
for r in C4_REWARDS:
    m = (r * ATT[r]['W'])[:, :C4_NFIT].ravel(); x0 = ATT[r]['x0'][:, :C4_NFIT].ravel()
    a = ATT[r]['A'][:, :C4_NFIT].ravel(); idx = np.digitize(m, _e) - 1
    print(f'  s={r}: ' + '  '.join(f'{x0[idx == k].mean()/r:.2f}/{a[idx == k].mean():.2f}'
                                   for k in range(len(_e) - 1) if (idx == k).sum() > 2000))

### C3 note — the slope is the *realized* drive

The attractor is $X = x_0 W$, and $x_0$ relaxes toward $s$ only on rewarded trials, so the slope is the
drive $\langle x_0\rangle = s\,P(\text{optimal})$. During learning it is below $s$ (binned by $m$,
$\langle x_0\rangle/s$ and $P$ agree to within 0.01, printed above). After learning $P\approx0.99$ at
every reward, so the steady-state slope equals $s$ to three decimals. Main Fig. 6f shows the
steady-state slopes.

## Panel C4 — the dynamics along the attractor predict the learning curves

The PC update $\Delta W = \beta\,x_0\,(X - x_0 W)$ vanishes on the attractor, so transport along the
line comes from the deviation $X-x_0W$ that activity noise creates. That deviation is reward-correlated
— the network is rewarded when noise pushed $X>0$, and reward raises the gate $x_0$ — so learning is
noise-driven credit assignment, gated by $x_0$. It stops when the network chooses the paying arm on
almost every trial: reward then no longer depends on the noise, $x_0$ stays at $s$, and the
correlation vanishes.

We measure $v(W) = \langle \Delta W \mid W\rangle/\Delta t$ and $D(W) = \mathrm{Var}(\Delta W\mid W)/2\Delta t$
at $\Delta t = 400$ trials. The drift changes sharply at the decision boundary, so $W$ is binned at 0.01
for $-0.05<W<0.1$ and 0.05 elsewhere. The third panel shows the drift of the predicted margin
$m=\text{slope}\cdot W$, which is where the asymptote comes from.

In [ ]:
W_EDGES = np.unique(np.concatenate([np.arange(-0.70, -0.05, 0.05), np.arange(-0.05, 0.10, 0.01),
                                    np.arange(0.10, 1.0001, 0.05)]).round(4))


def drift_diffusion(W, edges=W_EDGES, lag=LAG, lag_steps=LAG_STEPS, min_n=1500):
    """Binned drift v(W) and diffusion D(W), per trial, from increments over `lag_steps` trials."""
    W0 = W[:-lag].ravel(); d = (W[lag:] - W[:-lag]).ravel()
    ctr = (edges[1:] + edges[:-1]) / 2; idx = np.digitize(W0, edges) - 1
    v = np.full(len(ctr), np.nan); D = np.full(len(ctr), np.nan); n = np.zeros(len(ctr), int)
    for k in range(len(ctr)):
        sel = idx == k; n[k] = sel.sum()
        if n[k] >= min_n:
            v[k] = d[sel].mean() / lag_steps; D[k] = d[sel].var() / (2 * lag_steps)
    return ctr, v, D, n


def drift_in_m(r, edges=None, min_n=1500):
    """Drift of the margin coordinate m = slope*W, binned in m (fit seeds)."""
    edges = np.array([-0.3, -0.1, -0.05, -0.03, -0.02, -0.015, -0.01, -0.005, 0, 0.005, 0.01, 0.015,
                      0.02, 0.025, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2, 0.4]) if edges is None else edges
    m = ATT[r]['slope'] * ATT[r]['W'][:, :C4_NFIT]
    m0 = m[:-LAG].ravel(); dm = ((m[LAG:] - m[:-LAG]) / LAG_STEPS).ravel()
    idx = np.digitize(m0, edges) - 1; ctr = (edges[1:] + edges[:-1]) / 2
    return ctr, np.array([dm[idx == k].mean() if (idx == k).sum() >= min_n else np.nan for k in range(len(ctr))])


DD = {r: drift_diffusion(ATT[r]['W'][:, :C4_NFIT]) for r in C4_REWARDS}


def _dd_panels(axes):
    for r in C4_REWARDS:
        ctr, v, D, _ = DD[r]; g = np.isfinite(v)
        axes[0].plot(ctr[g], v[g] * 1e5, color=reward_color_abs(r), lw=1.0, label=f'{r}')
        axes[1].plot(ctr[g], D[g] * 1e8, color=reward_color_abs(r), lw=1.0, label=f'{r}')
    for ax in axes[:2]:
        ax.axhline(0, color='0.8', lw=0.5, zorder=0); ax.axvline(0, color='0.8', lw=0.5, zorder=0)
    nature_ax(axes[0], '$W = w_2-w_1$', r'Drift $v(W)$' '\n' r'($\times 10^{-5}$/trial)')
    nature_ax(axes[1], '$W = w_2-w_1$', r'Diffusion $D(W)$' '\n' r'($\times 10^{-8}$/trial)')


# main Fig. 6g: drift and diffusion
fig, axes = plt.subplots(1, 2, figsize=(4.3, 1.5), gridspec_kw=dict(wspace=0.5))
_dd_panels(axes); side_legend(axes[1], title='Reward')
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_drift_diffusion.svg', dpi=300, bbox_inches='tight')
plt.show()

# three-panel version (companion paper Fig. 7): adds the drift of the margin m
fig, axes = plt.subplots(1, 3, figsize=(6.6, 1.55), gridspec_kw=dict(wspace=0.65))
_dd_panels(axes)
for r in C4_REWARDS:
    cm, vm = drift_in_m(r); gm = np.isfinite(vm)
    axes[2].plot(cm[gm], vm[gm] * 1e5, '-o', ms=1.6, mew=0, color=reward_color_abs(r), lw=0.9, label=f'{r}')
axes[2].axhline(0, color='0.8', lw=0.5, zorder=0); axes[2].axvline(0, color='0.8', lw=0.5, zorder=0)
nature_ax(axes[2], r'$m = \mathrm{slope}\times W$', r'Drift of $m$' '\n' r'($\times 10^{-5}$/trial)')
axes[2].set_xlim(-0.1, 0.1); side_legend(axes[2], title='Reward')
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_drift_diffusion_margin.svg', dpi=300, bbox_inches='tight')
plt.show()

for r in C4_REWARDS:
    ctr, v, D, _ = DD[r]
    wr = (ctr < -0.05) & (ctr > -0.3) & np.isfinite(v); rt = (ctr > 0.1) & np.isfinite(v)
    print(f's={r}: wrong side v = {np.nanmean(v[wr])*1e5:.3f}e-5 (v/s {np.nanmean(v[wr])*1e5/r:.2f}e-5), '
          f'D = {np.nanmean(D[wr])*1e8:.2f}e-8;  beyond the boundary (W>0.1) v = {np.nanmean(v[rt])*1e5:+.3f}e-5, '
          f'D = {np.nanmean(D[rt])*1e8:.2f}e-8')
print('\nwhere the drift of m stops (fine bins in m):')
fine = np.arange(-0.05, 0.06, 0.0025)
for r in C4_REWARDS:
    c, vm = drift_in_m(r, edges=fine, min_n=300)
    ref = np.nanmean(vm[c < -0.01]); g = np.isfinite(vm) & (c > 0)
    print(f'  s={r}: wrong-side drift of m {ref*1e5:.3f}e-5;  10% of it at m = {c[g][np.argmax(vm[g] < 0.1*ref)]:.4f};'
          f'  zero at m = {c[g][np.argmax(vm[g] <= 0)]:.4f}')

### C4b — behaviour is one fixed function of the attractor coordinate

Pooled per seed (never seed-average $W$ and $P$ first: $P$ is a sigmoid of the margin, so that is
Jensen-biased), $P(\text{optimal})$ against $m=\text{slope}\cdot W$ falls on one curve for all rewards.
The curve is asymmetric because the gate shrinks the margin when the network misses: the margin the
weight produces is $x_0W = s\,P\,W = P\,m$, so $P(m)$ solves $P = \Phi(P\,m/\sigma_{\rm OU})$, with no
free parameter (checked below). Against the measured margin $X$ the choice is a probit of width one
noise increment, $\sqrt2\,\eta$.

In [ ]:
from scipy.stats import norm
from scipy.optimize import brentq

SIGMA_DEC = np.sqrt(2) * ETA_A
M_EDGES = np.linspace(-0.06, 0.06, 49)


def pooled_curve(coord, action, edges, min_n=500):
    c = (edges[1:] + edges[:-1]) / 2; idx = np.digitize(coord, edges) - 1
    y = np.array([action[idx == k].mean() if (idx == k).sum() >= min_n else np.nan for k in range(len(c))])
    return c, y


def best_width(c, y):
    g = np.geomspace(0.0005, 0.3, 1500)
    rms = [np.sqrt(np.nanmean((y - norm.cdf(c / s)) ** 2)) for s in g]
    k = int(np.argmin(rms)); return g[k], rms[k]


# shared behaviour curve, FIT SEEDS ONLY (held-out seeds test C4c)
m_fit = np.concatenate([(ATT[r]['slope'] * ATT[r]['W'])[:, :C4_NFIT].ravel() for r in C4_REWARDS])
a_fit = np.concatenate([ATT[r]['A'][:, :C4_NFIT].ravel() for r in C4_REWARDS])
MC, PC_ = pooled_curve(m_fit, a_fit, M_EDGES)
_ok = np.isfinite(PC_); MC, PC_ = MC[_ok], PC_[_ok]
P_of_margin = lambda m: np.interp(m, MC, PC_, left=PC_[0], right=PC_[-1])

fig, axes = plt.subplots(1, 2, figsize=(4.6, 1.75), gridspec_kw=dict(wspace=0.55))
for r in C4_REWARDS:
    c, y = pooled_curve((ATT[r]['slope'] * ATT[r]['W']).ravel(), ATT[r]['A'].ravel(), np.linspace(-0.06, 0.06, 25))
    g = np.isfinite(y); axes[0].plot(c[g], y[g], 'o', ms=2.6, color=reward_color_abs(r), mew=0)
axes[0].plot(MC, PC_, 'k-', lw=1.0, zorder=3)
nature_ax(axes[0], r'$m = \mathrm{slope}\times W$', 'P(optimal)'); axes[0].set_xlim(-0.06, 0.06)
XE = np.linspace(-6 * SIGMA_DEC, 6 * SIGMA_DEC, 25)
for r in C4_REWARDS:
    c, y = pooled_curve(ATT[r]['X'].ravel(), ATT[r]['A'].ravel(), XE)
    g = np.isfinite(y); axes[1].plot(c[g], y[g], 'o', ms=2.6, color=reward_color_abs(r), mew=0, label=f'{r}')
xx = np.linspace(-6 * SIGMA_DEC, 6 * SIGMA_DEC, 200)
axes[1].plot(xx, norm.cdf(xx / SIGMA_DEC), 'k-', lw=1.2, zorder=3); axes[1].set_xlim(XE[0], XE[-1])
nature_ax(axes[1], '$X = x_2-x_1$  (measured margin)', 'P(optimal)', nx=3)
for ax in axes:
    ax.axhline(0.5, color='0.85', lw=0.4, zorder=0); ax.axvline(0, color='0.85', lw=0.4, zorder=0)
    ax.set_ylim(-0.02, 1.02)
side_legend(axes[1], title='Reward')
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_collapse.svg', dpi=300, bbox_inches='tight')
plt.show()

X_all = np.concatenate([ATT[r]['X'].ravel() for r in C4_REWARDS])
A_all = np.concatenate([ATT[r]['A'].ravel() for r in C4_REWARDS])
cx, yx = pooled_curve(X_all, A_all, np.linspace(-6 * SIGMA_DEC, 6 * SIGMA_DEC, 49)); gx = np.isfinite(yx)
w_dec = best_width(cx[gx], yx[gx])
print(f'width against X: predicted sqrt(2) eta = {SIGMA_DEC:.5f}, best fit {w_dec[0]:.5f}')
for r in C4_REWARDS:
    c, y = pooled_curve((ATT[r]['slope'] * ATT[r]['W'])[:, :C4_NFIT].ravel(), ATT[r]['A'][:, :C4_NFIT].ravel(), M_EDGES)
    print(f'  width of P(m) at s={r}: {best_width(c[np.isfinite(y)], y[np.isfinite(y)])[0]:.4f}')
P_sc = lambda m: brentq(lambda p: p - norm.cdf(p * m / SIGMA_OU), 1e-9, 1.0)
g = np.abs(MC) < 0.05
print(f'self-consistent P = Phi(P m / sigma_OU) vs pooled P(m): RMS '
      f'{np.sqrt(np.mean((np.array([P_sc(m) for m in MC[g]]) - PC_[g])**2)):.4f} over |m| < 0.05 (no free parameter)')
print(f'pooled P(m = 0.01, 0.02, 0.03) = {P_of_margin(0.01):.3f}, {P_of_margin(0.02):.3f}, {P_of_margin(0.03):.3f}')

### C4c — the reconstruction (main Fig. 6h)

C3 gives the slope, C4a gives $v$ and $D$, C4b gives $P(m)$. Integrating the Fokker–Planck equation
forward from the held-out seeds' initial $W$ predicts the whole learning curve for each reward, with no
fitted behavioural parameters. Coefficients come from seeds 0–119; the curves they are tested against
come from seeds 120–199. Upwind finite-volume scheme (conservative, positivity-preserving; centred
differences oscillate on the spiky initial density).

In [ ]:
def fp_integrate(grid, v, D, rho0, t_eval, cfl=0.4):
    """d(rho)/dt = -d(v rho)/dW + d2(D rho)/dW2; donor-cell advection, centred diffusion, zero-flux ends."""
    h = grid[1] - grid[0]
    dt = cfl * min(h * h / (2 * max(D.max(), 1e-14)), h / max(np.abs(v).max(), 1e-14))
    vf = 0.5 * (v[1:] + v[:-1]); rho = rho0.copy()
    want = sorted(t_eval); out, wi, T = {}, 0, 0.0
    while wi < len(want):
        while wi < len(want) and T >= want[wi] - dt / 2:
            out[want[wi]] = rho.copy(); wi += 1
        if wi >= len(want):
            break
        adv = np.where(vf > 0, vf * rho[:-1], vf * rho[1:])
        dif = -(D[1:] * rho[1:] - D[:-1] * rho[:-1]) / h
        F = np.zeros(len(rho) + 1); F[1:-1] = adv + dif
        rho = rho - dt * (F[1:] - F[:-1]) / h; T += dt
    return out, dt


GRID = np.arange(-1.2, 1.2, 0.0025)
T_EVAL = [2000, 5000, 10_000, 20_000, 40_000, 80_000, 150_000, 300_000]


def fp_run(r, times, lag=LAG, lag_steps=LAG_STEPS):
    ctr, v, D, _ = drift_diffusion(ATT[r]['W'][:, :C4_NFIT], lag=lag, lag_steps=lag_steps)
    g = np.isfinite(v) & np.isfinite(D)
    v_g = np.interp(GRID, ctr[g], v[g]); D_g = np.maximum(np.interp(GRID, ctr[g], D[g]), 1e-12)
    W0 = ATT[r]['W'][0, C4_NFIT:]
    rho0 = np.exp(-0.5 * ((GRID[:, None] - W0[None, :]) / 0.01) ** 2).sum(1)
    rho0 /= rho0.sum() * (GRID[1] - GRID[0])
    return fp_integrate(GRID, v_g, D_g, rho0, times)


def measured(r, t):
    Ah = ATT[r]['A'][:, C4_NFIT:]; half = 500 // C4_STRIDE          # +/-500 trials around t
    i0 = min(max(0, t // C4_STRIDE - half), Ah.shape[0] - 2 * half)
    return Ah[i0:i0 + 2 * half].mean()


pred, meas = {}, {}
T_LINE = sorted(set([1000] + list(range(5000, 300_001, 5000)) + T_EVAL))
fig, ax = plt.subplots(figsize=(2.7, 1.75))
for r in C4_REWARDS:
    rh, FP_DT = fp_run(r, T_LINE)
    line = np.array([np.trapezoid(rh[t] * P_of_margin(ATT[r]['slope'] * GRID), GRID) for t in T_LINE])
    pred[r] = np.array([line[T_LINE.index(t)] for t in T_EVAL])
    meas[r] = np.array([measured(r, t) for t in T_EVAL])
    ax.plot(T_EVAL, meas[r], 'o', ms=3, color=reward_color_abs(r), mew=0, zorder=3)
    ax.plot(T_LINE, line, '-', lw=1.0, color=reward_color_abs(r), zorder=2)
nature_ax(ax, 'Trial', 'P(optimal)')
ax.set_xlim(0, 300_000); ax.set_ylim(0.48, 1.01)
ax.xaxis.set_major_locator(mticker.MultipleLocator(100_000))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x else '0'))
side_legend(ax, handles=
    [plt.Line2D([], [], color=reward_color_abs(r), lw=1.4, label=f'{r}') for r in C4_REWARDS]
    + [plt.Line2D([], [], color='0.3', marker='o', ms=3, lw=0, label='measured'),
       plt.Line2D([], [], color='0.3', lw=1.0, label='Fokker–Planck')], title='Reward')
plt.tight_layout()
plt.savefig(f'{SVG_DIR}/f6ii_mag_fp_reconstruction.svg', dpi=300, bbox_inches='tight')
plt.show()

print('held-out learning curves: measured / Fokker-Planck predicted')
print(f"{'s':>5} " + ' '.join(f'{t//1000:>10}k' for t in T_EVAL))
for r in C4_REWARDS:
    print(f'{r:5.2f} ' + ' '.join(f'{m:.3f}/{p:.3f}' for m, p in zip(meas[r], pred[r])))
err = np.concatenate([pred[r] - meas[r] for r in C4_REWARDS])
print(f'timestep {FP_DT:.1f} trials;  RMS = {np.sqrt(np.mean(err**2)):.4f}, max |error| = {np.abs(err).max():.4f}')

### C4d — the two-factor law

The realized drive enters twice — as the plasticity gate and as the attractor slope — so at matched
behaviour the weight drift should be linear in the slope and the margin velocity quadratic. Matched
behaviour here is $-0.04 < m < -0.01$ ($P\approx0.23$ at every reward). The noise-cancellation sweep
(companion paper Table 2) is in `companion_paper/`.

In [ ]:
print(f"{'s':>5} {'slope':>6} {'drive':>6} {'P':>6} {'v (1e-5)':>9} {'slope*v (1e-5)':>15}")
v0, dr0 = [], []
for r in C4_REWARDS:
    W = ATT[r]['W'][:, :C4_NFIT]; sl = ATT[r]['slope']; m = (sl * W)[:-LAG].ravel()
    dW = ((W[LAG:] - W[:-LAG]) / LAG_STEPS).ravel()
    x0 = ATT[r]['x0'][:, :C4_NFIT][:-LAG].ravel(); a = ATT[r]['A'][:, :C4_NFIT][:-LAG].ravel()
    sel = (m > -0.04) & (m < -0.01)
    v0.append(dW[sel].mean()); dr0.append(x0[sel].mean())
    print(f'{r:5.2f} {sl:6.3f} {dr0[-1]:6.3f} {a[sel].mean():6.3f} {v0[-1]*1e5:9.3f} {sl*v0[-1]*1e5:15.3f}')
sl_a = np.array([ATT[r]['slope'] for r in C4_REWARDS]); v0 = np.array(v0)
ex = lambda x, y: np.polyfit(np.log(x), np.log(y), 1)[0]
print(f'\n  v       ~ slope^{ex(sl_a, v0):.2f}   (predicted 1: the plasticity gate)')
print(f'  slope*v ~ slope^{ex(sl_a, sl_a*v0):.2f}   (predicted 2: gate x geometry)')

### C4e — is the one-dimensional reduction legitimate?

$P(t)=\int P(m)\rho(W,t)\,dW$ is a smoothed functional of the density, so C4c alone is not enough.
Four checks: (1) the density itself against held-out histograms (KS against the $n=80$ critical value);
(2) insensitivity to the coarse-graining lag; (3) time homogeneity of the drift, against a noise floor
from two random halves of the seeds in one window; (4) residual dependence on recent reward rate at
matched $W$, and the increment autocorrelation that sets the coarse-graining time.

In [ ]:
DENS_R = [0.2, 0.5, 1.0]
DENS_T = [10_000, 40_000, 100_000, 300_000]


def ks_stat(rho, sample):
    h = GRID[1] - GRID[0]; cdf = np.cumsum(rho) * h; cdf /= cdf[-1]; s = np.sort(sample)
    return float(np.abs(np.interp(s, GRID, cdf) - np.arange(1, len(s) + 1) / len(s)).max())


fig, axes = plt.subplots(len(DENS_R), len(DENS_T), figsize=(6.4, 2.9), sharex=True,
                         gridspec_kw=dict(hspace=0.55, wspace=0.35))
bins = np.arange(-1.0, 1.0001, 0.04); ks_all = []
for i, r in enumerate(DENS_R):
    rh, _ = fp_run(r, DENS_T); Wh = ATT[r]['W'][:, C4_NFIT:]
    for j, t in enumerate(DENS_T):
        ax = axes[i, j]; mw = Wh[min(t // C4_STRIDE, Wh.shape[0]) - 1]
        ax.hist(mw, bins=bins, density=True, color=reward_color_abs(r), alpha=0.45, lw=0)
        k = int(round(0.04 / (GRID[1] - GRID[0])))
        ax.plot(GRID, np.convolve(rh[t], np.ones(k) / k, 'same'), 'k-', lw=0.9)   # smoothed to the bin width
        kv = ks_stat(rh[t], mw); ks_all.append(kv)
        ax.set_title(f'$s$={r}, {t//1000}k   KS={kv:.2f}', fontsize=6.5, pad=2)
        nature_ax(ax, '$W$' if i == len(DENS_R) - 1 else '', 'density' if j == 0 else '', nx=3, ny=3)
        ax.set_xlim(-1.0, 1.0)
plt.savefig(f'{SVG_DIR}/f6ii_mag_fp_density.svg', dpi=300, bbox_inches='tight')
plt.show()
print(f'CHECK 1 - density: KS max {max(ks_all):.3f}, mean {np.mean(ks_all):.3f} '
      f'(n=80 critical value {1.36/np.sqrt(80):.3f})')

print('\nCHECK 2 - lag insensitivity:  lag  P RMS  FP sd/measured sd @300k')
for lag_s in (100, 200, 400, 800, 1600, 3200):
    errs, sdr = [], []
    for r in C4_REWARDS:
        rh, _ = fp_run(r, T_EVAL, lag=lag_s // C4_STRIDE, lag_steps=lag_s)
        pr = np.array([np.trapezoid(rh[t] * P_of_margin(ATT[r]['slope'] * GRID), GRID) for t in T_EVAL])
        errs.append(pr - meas[r])
        rf = rh[T_EVAL[-1]]; fm = np.trapezoid(rf * GRID, GRID)
        sdr.append(np.sqrt(np.trapezoid(rf * (GRID - fm) ** 2, GRID)) / ATT[r]['W'][-1, C4_NFIT:].std())
    e = np.concatenate(errs)
    print(f'  {lag_s:5d}  {np.sqrt((e**2).mean()):.4f}  {np.mean(sdr):.3f}')

print('\nCHECK 3 - time homogeneity, first vs last third of 0-150k:')
_rng = np.random.default_rng(0)
for r in C4_REWARDS:
    W = ATT[r]['W'][:15_000, :C4_NFIT]; n = W.shape[0]
    _, ve, _, _ = drift_diffusion(W[:n // 3], min_n=400); _, vl, _, _ = drift_diffusion(W[2 * n // 3:], min_n=400)
    g = np.isfinite(ve) & np.isfinite(vl)
    rel = np.median(np.abs(ve[g] - vl[g])) / np.max(np.abs(np.concatenate([ve[g], vl[g]])))
    sh = _rng.permutation(C4_NFIT); early = W[:n // 3]
    _, va, _, _ = drift_diffusion(early[:, sh[:C4_NFIT // 2]], min_n=400)
    _, vb, _, _ = drift_diffusion(early[:, sh[C4_NFIT // 2:]], min_n=400)
    g2 = np.isfinite(va) & np.isfinite(vb)
    nf = np.median(np.abs(va[g2] - vb[g2])) / np.max(np.abs(np.concatenate([va[g2], vb[g2]])))
    print(f'  s={r}: |dv|/max|v| {rel:.3f}, noise floor {nf:.3f} -> '
          f'{"within sampling noise" if rel <= nf else f"{rel/nf:.1f}x the noise floor"}')

print('\nCHECK 4a - autocorrelation of the dW increments (lag in trials: r)')
for r in (0.5, 1.0):
    inc = np.diff(ATT[r]['W'][:, :C4_NFIT], axis=0); inc = inc - inc.mean(0, keepdims=True)
    print(f'  s={r}: ' + '  '.join(f'{k*C4_STRIDE}: {(inc[:-k]*inc[k:]).mean()/inc.var():.3f}' for k in (1, 2, 4, 10, 40)))

print('\nCHECK 4b - matched W on the wrong side (-0.3 < W < -0.1): drift at low vs high recent reward rate')
_H = 1000 // C4_STRIDE
for r in (0.3, 0.5, 1.0):
    W = ATT[r]['W'][:, :C4_NFIT]; A = ATT[r]['A'][:, :C4_NFIT].astype(np.float64)
    cs = np.cumsum(np.vstack([np.zeros((1, A.shape[1])), A]), axis=0); rr = (cs[_H:-1] - cs[:-_H - 1]) / _H
    Wv = W[_H:-LAG].ravel(); dv = ((W[_H + LAG:] - W[_H:-LAG]) / LAG_STEPS).ravel(); rv = rr[:-LAG].ravel()
    sel = (Wv > -0.3) & (Wv < -0.1); q = np.percentile(rv[sel], [33, 67])
    vlo, vhi = dv[sel][rv[sel] < q[0]].mean(), dv[sel][rv[sel] > q[1]].mean()
    print(f'  s={r}: v {vlo*1e5:.3f}e-5 vs {vhi*1e5:.3f}e-5  -> (hi-lo)/mean = {(vhi-vlo)/abs((vhi+vlo)/2):+.2f}')

## Notes and caveats

**The chain, as measured.** Reward magnitude $s$ sets the realized drive
$\langle x_0\rangle = s\,P(\text{optimal})$ (C3). That one quantity is both the attractor slope and the
gate on plasticity, so the margin velocity at matched behaviour scales as slope$^{2}$ (C4d) and the
noise cancels from the per-network behavioural rate (companion paper Table 2). The population rate scales
less steeply (rate ∝ $s^{1.09}$, Panel B) because it also includes the travel from the initial weight
to the decision boundary, at a speed with one factor of drive.

**Why the asymptote is the same at every reward.** The drift of $m$ falls to zero at $m\approx0.02$
for every reward (C4a), where $P(m)\approx0.99$ (C4b): once the network chooses the paying arm on
almost every trial, reward no longer depends on the noise and the gate stops correlating with it.
The stopping point is fixed in the margin, in units of the noise, not in the weight.

**Caveats on this notebook's own numbers.**

* $\eta = 0.003$ is between the det sweep's grid levels; $(\alpha,\beta)$ is a top-scoring sweep cell
  at $\eta=10^{-2}$ (see CONFIG).
* Time homogeneity (C4e check 3) holds within sampling noise at $s=0.2$ and $0.3$ but not at
  $s\ge0.5$ (1.9–4.8× the noise floor), and the predicted density at 300k is ~13% wider than measured.
  $v(W)$ is a good population average but not an exact Markov law.